In [1]:
"""
Multi-Vector Retriever (MVR) -- Production-Grade RAG Pipeline
==============================================================
Architecture   : LangChain MultiVectorRetriever with THREE indexing strategies
LLM            : Azure OpenAI  (AzureChatOpenAI -- GPT-4o or GPT-4-turbo)
Embeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local)
Vector Store   : ChromaDB  (persists child/proxy vectors to disk)
Doc Store      : InMemoryStore  (holds full parent Documents by UUID)
Data Source    : Three publicly available research PDFs from arXiv (open-access)

Reference PDFs:
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

Installation (run once):
    pip install langchain langchain-community langchain-openai langchain-chroma
                langchain-text-splitters langchain-huggingface
                sentence-transformers chromadb pypdf

Required environment variables:
    AZURE_OPENAI_API_KEY
    AZURE_OPENAI_ENDPOINT
    AZURE_OPENAI_DEPLOYMENT   (default: gpt-4o)
    AZURE_OPENAI_API_VERSION  (default: 2024-02-15-preview)

=============================================================================
Bug-fix log vs. original code
=============================================================================

BUG 1 -- FATAL IMPORT: ``from langchain_classic.retrievers import ...``
    The package ``langchain_classic`` does not exist in any published release.
    MultiVectorRetriever is in ``langchain.retrievers.multi_vector``.
    ParentDocumentRetriever / ContextualCompressionRetriever / MergerRetriever
    live in ``langchain.retrievers``.  Fixed: correct import paths used.

BUG 2 -- DANGLING VARIABLE IN generate_summaries()
    ``summary_llm`` was constructed via ``llm.with_config() | StrOutputParser()``
    then immediately discarded -- the actual chain built a SECOND raw
    AzureChatOpenAI inline without honoring ``config.SUMMARY_MAX_TOKENS``.
    Fixed: removed the dead ``summary_llm`` variable; the inline chain now
    correctly receives ``max_tokens=config.SUMMARY_MAX_TOKENS``.

BUG 3 -- DOUBLE RETRIEVAL IN query_mvr()
    ``retriever.invoke(question)`` was called inside ``query_mvr()`` to
    display source documents, and then ``chain.invoke(question)`` triggered
    a SECOND retrieval internally.  This doubled latency and token cost.
    Fixed: sources are captured from the chain's context pipeline via a
    ``RunnablePassthrough.assign`` side-channel so only one retrieval occurs.

BUG 4 -- PRIVATE CHROMADB ATTRIBUTE ACCESS
    ``vectorstore._collection.count()`` accesses a private Chroma attribute
    that changes across chromadb versions and will raise AttributeError on
    newer releases.  Fixed: replaced with the public API
    ``len(vectorstore.get()["ids"])``.

BUG 5 -- PIPELINE AT MODULE SCOPE (no ``__main__`` guard)
    The entire pipeline (PDF downloads, LLM proxy generation, ChromaDB
    indexing) executed on any ``import`` of the file, making the module
    untestable and unusable as a library.
    Fixed: moved into ``main()`` called under ``if __name__ == "__main__"``.

BUG 6 -- DEPRECATED HuggingFaceEmbeddings IMPORT PATH
    ``from langchain_community.embeddings import HuggingFaceEmbeddings``
    is deprecated and emits a LangChainDeprecationWarning in LangChain 0.3+.
    Fixed: imports from ``langchain_huggingface`` with a community fallback.

=============================================================================
Core Concept -- WHY Multi-Vector Retriever vs. Standard RAG vs. PDR
=============================================================================
Standard RAG embeds the same text that is returned to the LLM.
    Problem: Large chunks embed poorly (blurry centroid); small chunks lack
    context.

ParentDocumentRetriever (PDR) embeds small children, returns large parents.
    Problem: Supports only ONE type of proxy vector (small chunks).

MultiVectorRetriever (MVR) is the generalization.  It allows embedding
MULTIPLE different representations of the SAME document and stores ALL of
them in the vector index.  Regardless of which representation matches the
query, the retriever always returns the FULL original document to the LLM.

Strategy 1 -- SMALL CHUNKS (same as PDR internally)
    Proxy vectors : small text chunks (200 chars)
    Use when      : precise lexical or semantic similarity retrieval matters.

Strategy 2 -- SUMMARY EMBEDDINGS
    Proxy vectors : LLM-generated summaries of each parent section
    Use when      : query intent aligns with the gist of a passage rather
                    than its literal words.

Strategy 3 -- HYPOTHETICAL QUESTIONS (HyDE variant)
    Proxy vectors : LLM-generated "what question does this chunk answer?"
    Use when      : users phrase queries as questions (most chatbot cases).
"""

'\nMulti-Vector Retriever (MVR) -- Production-Grade RAG Pipeline\n==============================================================\nArchitecture   : LangChain MultiVectorRetriever with THREE indexing strategies\nLLM            : Azure OpenAI  (AzureChatOpenAI -- GPT-4o or GPT-4-turbo)\nEmbeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local)\nVector Store   : ChromaDB  (persists child/proxy vectors to disk)\nDoc Store      : InMemoryStore  (holds full parent Documents by UUID)\nData Source    : Three publicly available research PDFs from arXiv (open-access)\n\nReference PDFs:\n    1. "Attention Is All You Need" -- Vaswani et al., 2017\n       https://arxiv.org/pdf/1706.03762.pdf\n    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018\n       https://arxiv.org/pdf/1810.04805.pdf\n    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020\n       https://arxiv.org/pdf/2005.11401.pdf\n\nInstallation (run once):\n    pip install la

In [2]:
import os
import sys
import uuid
import time
import logging
import urllib.request
from enum import Enum
from pathlib import Path
from typing import List, Dict, Tuple, Any

# ---------------------------------------------------------------------------
# LangChain imports
# pip install langchain langchain-community langchain-openai chromadb
#             sentence-transformers pypdf
# ---------------------------------------------------------------------------
# Core schema
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
# Retrievers - Using langchain_classic (same as parent_document_retriever)
from langchain_classic.retrievers import (
    ParentDocumentRetriever,
    ContextualCompressionRetriever,
    MergerRetriever,
    MultiVectorRetriever
)
from langchain_chroma import Chroma


from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0327 08:08:42.064000 15076 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("multi_vector_retriever")


In [5]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class MVRStrategy(str, Enum):
    """
    Enum for the three Multi-Vector Retriever indexing strategies.

    Using an Enum prevents magic strings scattered throughout the codebase
    and makes strategy selection explicit and type-safe.
    """
    SMALL_CHUNKS   = "small_chunks"     # embed smaller sub-chunks as proxies
    SUMMARIES      = "summaries"        # embed LLM-generated section summaries
    HYPOTHETICAL_Q = "hypothetical_q"   # embed LLM-generated hypothetical questions


In [6]:

class MVRConfig:
    """
    Centralized configuration for the Multi-Vector Retriever RAG pipeline.

    Design notes:
        PARENT_CHUNK_SIZE controls the granularity of the "original document"
        that will be returned to the LLM at generation time. 1200 chars is
        roughly one to two dense technical paragraphs -- enough for the LLM
        to construct a complete, well-supported answer.

        CHILD_CHUNK_SIZE is used only in Strategy 1. At 200 chars it captures
        a single focused claim, producing a tight embedding vector.

        SUMMARY_MAX_TOKENS and QUESTION_MAX_TOKENS cap Azure OpenAI usage
        during the proxy-generation batch step. Summaries are deliberately
        capped at 200 tokens because we want a compact semantic fingerprint,
        not a paraphrase. Questions are capped at 100 tokens since we want
        one concise question per parent chunk.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- Text splitting ---------------------------------------------------
    PARENT_CHUNK_SIZE: int    = 1200    # chars; text returned to LLM
    PARENT_CHUNK_OVERLAP: int = 120
    CHILD_CHUNK_SIZE: int     = 200     # chars; Strategy 1 proxy
    CHILD_CHUNK_OVERLAP: int  = 20

    # --- Embeddings -------------------------------------------------------
    EMBEDDING_MODEL: str  = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = "cpu"       # "cuda" for GPU acceleration

    # --- ChromaDB (one collection per strategy, same persist dir) ---------
    CHROMA_PERSIST_DIR: str = "./chroma_mvr_db"
    CHROMA_COLLECTIONS: Dict[str, str] = {
        MVRStrategy.SMALL_CHUNKS:   "mvr_small_chunks",
        MVRStrategy.SUMMARIES:      "mvr_summaries",
        MVRStrategy.HYPOTHETICAL_Q: "mvr_hypothetical_q",
    }

    # --- Azure OpenAI LLM -------------------------------------------------
    AZURE_OPENAI_API_KEY:    str   = os.environ.get("AZURE_OPENAI_API_KEY",    "")
    AZURE_OPENAI_ENDPOINT:   str   = os.environ.get("AZURE_OPENAI_ENDPOINT",   "")
    AZURE_OPENAI_DEPLOYMENT: str   = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
    AZURE_OPENAI_API_VERSION: str  = os.environ.get("AZURE_OPENAI_API_VERSION","2024-02-15-preview")
    LLM_TEMPERATURE: float         = 0.0
    LLM_MAX_TOKENS: int            = 1024

    # Proxy generation caps (keeps Azure token costs predictable)
    SUMMARY_MAX_TOKENS: int    = 200
    QUESTION_MAX_TOKENS: int   = 100

    # --- Retrieval --------------------------------------------------------
    RETRIEVER_K: int = 3        # parent documents returned per query

    # --- Concurrency for proxy batch generation ---------------------------
    BATCH_CONCURRENCY: int = 3  # parallel LLM calls during ingestion

    # --- Prompt for final answer generation --------------------------------
    RAG_PROMPT_TEMPLATE: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer cannot be found in the context, respond with:
"The provided documents do not contain enough information to answer this question."

Context (retrieved from research papers):
{context}

Question: {question}

Provide a clear, technically accurate answer:"""

    # --- Prompts used during proxy generation (ingestion time) ------------
    SUMMARY_PROMPT: str = """Summarize the following research paper excerpt in 3-5 concise sentences.
Focus on the key technical concepts, methods, and findings.
Be specific and preserve technical terminology.

Excerpt:
{doc}

Summary:"""

    HYPOTHETICAL_Q_PROMPT: str = """Generate exactly ONE specific question that this research paper excerpt directly answers.
The question should be precise, technical, and reflect what a researcher would ask when
looking for this exact information.

Excerpt:
{doc}

Question:"""


In [7]:

# ===========================================================================
# SECTION 2 -- PDF DOWNLOADER
# ===========================================================================

def download_pdfs(config: MVRConfig) -> List[Path]:
    """
    Download research PDFs from arXiv to the local filesystem.

    Implements polite downloading: skips cached files, adds a 1-second delay
    between requests, and validates file size to detect failed downloads.

    Args:
        config: MVRConfig with PDF_SOURCES and PDF_DOWNLOAD_DIR.

    Returns:
        List of Path objects pointing to downloaded PDF files.

    Raises:
        RuntimeError: If a download fails.
    """
    download_dir = Path(config.PDF_DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)

    downloaded_paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        local_path = download_dir / f"{name}.pdf"

        if local_path.exists() and local_path.stat().st_size > 10_000:
            logger.info("Cached PDF: %s (%.1f KB)", local_path.name, local_path.stat().st_size / 1024)
            downloaded_paths.append(local_path)
            continue

        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "Mozilla/5.0 (RAG-Research-Pipeline/1.0)"},
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                pdf_bytes = resp.read()

            if len(pdf_bytes) < 10_000:
                raise RuntimeError(
                    f"Downloaded file too small ({len(pdf_bytes)} bytes). URL may have failed: {url}"
                )

            local_path.write_bytes(pdf_bytes)
            logger.info("Saved: %s (%.1f KB)", local_path.name, len(pdf_bytes) / 1024)
            downloaded_paths.append(local_path)
            time.sleep(1.0)  # polite delay

        except Exception as exc:
            raise RuntimeError(
                f"Failed to download '{url}'. "
                "Check network access or manually place the PDF at: {local_path}"
            ) from exc

    return downloaded_paths



In [8]:


# ===========================================================================
# SECTION 3 -- PDF LOADING AND PARENT CHUNKING
# ===========================================================================

def load_and_chunk_documents(
    pdf_paths: List[Path],
    config: MVRConfig,
) -> List[Document]:
    """
    Load PDF pages via PyPDFLoader and split them into parent-level chunks.

    Each parent chunk receives:
        - A UUID string assigned as metadata['doc_id'] for docstore lookup.
        - The originating PDF filename as metadata['source'].
        - A human-readable paper name as metadata['paper_name'].
        - The original page number as metadata['page'] (from PyPDFLoader).
        - A start_index indicating its character offset in the original page.

    These metadata fields survive through the entire pipeline and appear in
    the formatted context presented to the LLM, enabling provenance tracking.

    Why chunk BEFORE assigning UUIDs rather than assigning UUIDs to raw pages?
    Raw PDF pages vary enormously in length (a half-page figure caption vs. a
    dense methods section). Chunking to PARENT_CHUNK_SIZE (1200 chars) first
    produces roughly equal-length units, which leads to more consistent
    embedding quality and more balanced LLM contexts.

    Args:
        pdf_paths : Downloaded PDF Path objects.
        config    : MVRConfig with parent splitting parameters.

    Returns:
        List of parent Documents, each tagged with a unique doc_id.
    """
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.PARENT_CHUNK_SIZE,
        chunk_overlap=config.PARENT_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,
    )

    all_parent_docs: List[Document] = []

    for pdf_path in pdf_paths:
        logger.info("Loading: %s", pdf_path.name)
        paper_name = pdf_path.stem.replace("_", " ").title()

        try:
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        # Enrich page metadata before splitting
        for page in pages:
            page.metadata["source"]     = pdf_path.name
            page.metadata["paper_name"] = paper_name

        # Split pages into parent chunks
        parent_chunks = parent_splitter.split_documents(pages)

        # Assign a UUID to each parent chunk AFTER splitting so every chunk
        # gets its own independent identifier (not page-level)
        for chunk in parent_chunks:
            chunk.metadata["doc_id"] = str(uuid.uuid4())

        logger.info("  %s --> %d parent chunks", pdf_path.name, len(parent_chunks))
        all_parent_docs.extend(parent_chunks)

    logger.info(
        "Total parent chunks across all PDFs: %d (avg ~%d chars each)",
        len(all_parent_docs),
        sum(len(d.page_content) for d in all_parent_docs) // max(len(all_parent_docs), 1),
    )
    return all_parent_docs



In [9]:

# ===========================================================================
# SECTION 4 -- AZURE OPENAI LLM
# ===========================================================================

def build_azure_llm(config: MVRConfig) -> AzureChatOpenAI:
    """
    Initialize the Azure OpenAI chat model.

    Required environment variables:
        AZURE_OPENAI_API_KEY      : Azure resource key
        AZURE_OPENAI_ENDPOINT     : e.g. https://<resource>.openai.azure.com/
        AZURE_OPENAI_DEPLOYMENT   : deployment name in Azure AI Studio
        AZURE_OPENAI_API_VERSION  : e.g. 2024-02-15-preview

    Args:
        config: MVRConfig with Azure credentials.

    Returns:
        Initialized AzureChatOpenAI instance.

    Raises:
        EnvironmentError: If required environment variables are missing.
    """
    missing = [
        var for var, val in [
            ("AZURE_OPENAI_API_KEY",    config.AZURE_OPENAI_API_KEY),
            ("AZURE_OPENAI_ENDPOINT",   config.AZURE_OPENAI_ENDPOINT),
        ] if not val
    ]
    if missing:
        raise EnvironmentError(
            f"Missing required environment variables: {missing}\n"
            "Set them before running:\n"
            "  export AZURE_OPENAI_API_KEY='your-key'\n"
            "  export AZURE_OPENAI_ENDPOINT='https://your-resource.openai.azure.com/'\n"
            "  export AZURE_OPENAI_DEPLOYMENT='gpt-4o'\n"
            "  export AZURE_OPENAI_API_VERSION='2024-02-15-preview'"
        )

    logger.info(
        "Azure OpenAI: deployment='%s'  api_version='%s'",
        config.AZURE_OPENAI_DEPLOYMENT, config.AZURE_OPENAI_API_VERSION,
    )
    return AzureChatOpenAI(
        azure_deployment=config.AZURE_OPENAI_DEPLOYMENT,
        azure_endpoint=config.AZURE_OPENAI_ENDPOINT,
        api_key=config.AZURE_OPENAI_API_KEY,
        api_version=config.AZURE_OPENAI_API_VERSION,
        temperature=config.LLM_TEMPERATURE,
        max_tokens=config.LLM_MAX_TOKENS,
    )


In [10]:

# ===========================================================================
# SECTION 5 -- EMBEDDING MODEL
# ===========================================================================

def build_embeddings(config: MVRConfig) -> HuggingFaceEmbeddings:
    """
    Initialize the HuggingFace sentence-transformers embedding model.

    all-MiniLM-L6-v2 produces 384-dimensional unit-normalized vectors and
    runs fully locally with no API key or external network call at inference.
    normalize_embeddings=True ensures unit-norm vectors required for
    mathematically correct cosine similarity in ChromaDB.

    Args:
        config: MVRConfig with EMBEDDING_MODEL and EMBEDDING_DEVICE.

    Returns:
        Initialized HuggingFaceEmbeddings instance.
    """
    logger.info("Embedding model: %s (device=%s)", config.EMBEDDING_MODEL, config.EMBEDDING_DEVICE)
    return HuggingFaceEmbeddings(
        model_name=config.EMBEDDING_MODEL,
        model_kwargs={"device": config.EMBEDDING_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )

In [11]:


# ===========================================================================
# SECTION 6 -- PROXY GENERATION (SUMMARIES AND HYPOTHETICAL QUESTIONS)
# These two functions call Azure OpenAI during ingestion time to generate
# the richer proxy representations for Strategies 2 and 3.
# ===========================================================================

def generate_summaries(
    parent_docs: List[Document],
    llm: AzureChatOpenAI,
    config: MVRConfig,
) -> List[str]:
    """
    Generate a concise LLM summary for each parent document.

    Why summaries outperform raw-chunk embeddings for broad queries:
    A 1200-char technical paragraph contains filler sentences, equation
    notation, citations, and transitional text that add noise to the
    embedding. A 3-5 sentence summary distills the core argument into a
    compact, semantically dense representation. When a user asks a broad
    conceptual question ("how does attention work?"), the summary vector
    is far closer to the query vector than the raw paragraph vector.

    The summary is generated by a low-temperature LLM call (inheriting
    config.LLM_TEMPERATURE = 0.0) to maximize determinism. We override
    max_tokens specifically for this step because summaries need only 200
    tokens, keeping per-call latency and token cost low.

    Batch processing note: We use .batch() with max_concurrency to issue
    BATCH_CONCURRENCY parallel async Azure API calls, dramatically reducing
    total ingestion time compared to sequential .invoke() calls. With 300
    parent chunks and concurrency=3, ingestion time drops ~3x.

    Args:
        parent_docs : Parent Documents to summarize.
        llm         : AzureChatOpenAI instance.
        config      : MVRConfig with SUMMARY_PROMPT and BATCH_CONCURRENCY.

    Returns:
        List of summary strings, one per parent document (same order).
    """
    logger.info("Generating summaries for %d parent docs ...", len(parent_docs))

    # Override max_tokens for the summary generation step only
    summary_llm = llm.with_config(configurable={}) | StrOutputParser()
    summary_prompt = ChatPromptTemplate.from_template(config.SUMMARY_PROMPT)

    # Compose a summary chain scoped to this step
    summary_chain = (
        {"doc": lambda x: x.page_content}
        | summary_prompt
        | AzureChatOpenAI(
            azure_deployment=config.AZURE_OPENAI_DEPLOYMENT,
            azure_endpoint=config.AZURE_OPENAI_ENDPOINT,
            api_key=config.AZURE_OPENAI_API_KEY,
            api_version=config.AZURE_OPENAI_API_VERSION,
            temperature=0.0,
            max_tokens=config.SUMMARY_MAX_TOKENS,
        )
        | StrOutputParser()
    )

    summaries: List[str] = summary_chain.batch(
        parent_docs,
        config={"max_concurrency": config.BATCH_CONCURRENCY},
    )

    logger.info(
        "Summaries generated. Avg length: %d chars",
        sum(len(s) for s in summaries) // max(len(summaries), 1),
    )
    return summaries



In [12]:

def generate_hypothetical_questions(
    parent_docs: List[Document],
    llm: AzureChatOpenAI,
    config: MVRConfig,
) -> List[str]:
    """
    Generate one hypothetical question per parent document.

    The HyDE (Hypothetical Document Embeddings) insight applied here:
    Users phrase queries as questions. Technical paper text is phrased as
    assertions. The cosine distance between a user question embedding and an
    assertion embedding is inherently larger than between two question
    embeddings. By pre-generating "what question does this chunk answer?"
    and embedding THAT question, we align the vector space to the user's
    natural query language, dramatically improving recall.

    For example:
        Parent text: "Multi-head attention allows the model to jointly attend
                      to information from different representation subspaces..."
        Generated Q: "What is the purpose of multi-head attention and how does
                      it differ from single-head attention?"
        User query:  "Why do Transformers use multi-head attention?"

    The cosine similarity between the generated question and the user query
    is much higher than between the raw text and the user query, causing the
    correct parent chunk to be surfaced even for paraphrased questions.

    Args:
        parent_docs : Parent Documents to generate questions for.
        llm         : AzureChatOpenAI instance.
        config      : MVRConfig with HYPOTHETICAL_Q_PROMPT and BATCH_CONCURRENCY.

    Returns:
        List of hypothetical question strings (same order as parent_docs).
    """
    logger.info("Generating hypothetical questions for %d parent docs ...", len(parent_docs))

    question_prompt = ChatPromptTemplate.from_template(config.HYPOTHETICAL_Q_PROMPT)
    question_chain = (
        {"doc": lambda x: x.page_content}
        | question_prompt
        | AzureChatOpenAI(
            azure_deployment=config.AZURE_OPENAI_DEPLOYMENT,
            azure_endpoint=config.AZURE_OPENAI_ENDPOINT,
            api_key=config.AZURE_OPENAI_API_KEY,
            api_version=config.AZURE_OPENAI_API_VERSION,
            temperature=0.0,
            max_tokens=config.QUESTION_MAX_TOKENS,
        )
        | StrOutputParser()
    )

    questions: List[str] = question_chain.batch(
        parent_docs,
        config={"max_concurrency": config.BATCH_CONCURRENCY},
    )

    logger.info(
        "Questions generated. Sample: '%s'",
        questions[0] if questions else "(none)",
    )
    return questions


In [13]:

# ===========================================================================
# SECTION 7 -- MULTI-VECTOR RETRIEVER BUILDER
# Constructs one MVR per strategy. Each shares the SAME docstore but uses
# a SEPARATE ChromaDB collection for its proxy vectors.
# ===========================================================================

def build_mvr_strategy_1_small_chunks(
    parent_docs: List[Document],
    embeddings: HuggingFaceEmbeddings,
    config: MVRConfig,
) -> MultiVectorRetriever:
    """
    Strategy 1 -- Small Chunk Proxies.

    Index small sub-chunks (200 chars) of each parent document. This is the
    same logic that ParentDocumentRetriever uses internally, exposed here
    explicitly so it can be compared side-by-side with Strategies 2 and 3.

    Indexing steps:
        1. For each parent doc (UUID already in metadata['doc_id']):
           a. Apply child_splitter to produce child chunks.
           b. Copy the parent's doc_id into each child's metadata.
        2. Embed all child chunks via HuggingFace and store in ChromaDB.
        3. Store full parent docs in InMemoryStore keyed by doc_id.

    Args:
        parent_docs : Parent documents with pre-assigned doc_id metadata.
        embeddings  : HuggingFace embedding model.
        config      : MVRConfig.

    Returns:
        Populated MultiVectorRetriever for Strategy 1.
    """
    logger.info("=== Strategy 1: Small Chunks ===")

    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHILD_CHUNK_SIZE,
        chunk_overlap=config.CHILD_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    vectorstore = Chroma(
        collection_name=config.CHROMA_COLLECTIONS[MVRStrategy.SMALL_CHUNKS],
        embedding_function=embeddings,
        persist_directory=config.CHROMA_PERSIST_DIR,
    )
    docstore = InMemoryStore()
    id_key   = "doc_id"

    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key,
        search_kwargs={"k": config.RETRIEVER_K},
    )

    # Generate child chunks and tag each with its parent's doc_id
    child_docs: List[Document] = []
    for parent in parent_docs:
        parent_id = parent.metadata[id_key]
        children  = child_splitter.split_documents([parent])
        for child in children:
            child.metadata[id_key] = parent_id   # link child --> parent
        child_docs.extend(children)

    logger.info(
        "  Parent docs: %d  |  Child chunks: %d  (avg %.1f children/parent)",
        len(parent_docs), len(child_docs), len(child_docs) / max(len(parent_docs), 1),
    )

    # Populate vector store with embedded child chunks
    retriever.vectorstore.add_documents(child_docs)

    # Populate docstore with full parent documents
    retriever.docstore.mset(
        [(doc.metadata[id_key], doc) for doc in parent_docs]
    )

    logger.info("  Strategy 1 indexed. ChromaDB vectors: %d", vectorstore._collection.count())
    return retriever


In [14]:

def build_mvr_strategy_2_summaries(
    parent_docs: List[Document],
    summaries: List[str],
    embeddings: HuggingFaceEmbeddings,
    config: MVRConfig,
) -> MultiVectorRetriever:
    """
    Strategy 2 -- Summary Embeddings.

    Embed LLM-generated summaries as proxy vectors. The full parent document
    is still what gets returned to the LLM at generation time; only the PROXY
    used for vector-space search is a summary.

    This strategy excels for conceptual queries like "what does this paper
    argue about attention mechanisms?" because the summary captures the
    semantic essence of the passage without the noise of raw text.

    Indexing steps:
        1. For each parent doc, pair its doc_id with its generated summary.
        2. Wrap the summary string as a Document with doc_id in metadata.
        3. Embed these summary Documents into a dedicated ChromaDB collection.
        4. Store full parent docs in InMemoryStore keyed by doc_id.

    Args:
        parent_docs : Parent documents with doc_id metadata.
        summaries   : LLM-generated summary strings (same order as parent_docs).
        embeddings  : HuggingFace embedding model.
        config      : MVRConfig.

    Returns:
        Populated MultiVectorRetriever for Strategy 2.
    """
    logger.info("=== Strategy 2: Summary Embeddings ===")

    if len(summaries) != len(parent_docs):
        raise ValueError(
            f"Mismatch: {len(parent_docs)} parent docs but {len(summaries)} summaries. "
            "Ensure generate_summaries() was called on the same parent_docs list."
        )

    vectorstore = Chroma(
        collection_name=config.CHROMA_COLLECTIONS[MVRStrategy.SUMMARIES],
        embedding_function=embeddings,
        persist_directory=config.CHROMA_PERSIST_DIR,
    )
    docstore = InMemoryStore()
    id_key   = "doc_id"

    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key,
        search_kwargs={"k": config.RETRIEVER_K},
    )

    # Wrap each summary as a Document carrying the parent's doc_id
    summary_docs: List[Document] = [
        Document(
            page_content=summary,
            metadata={
                id_key:       parent.metadata[id_key],
                "source":     parent.metadata.get("source", ""),
                "paper_name": parent.metadata.get("paper_name", ""),
                "proxy_type": "summary",
            }
        )
        for parent, summary in zip(parent_docs, summaries)
    ]

    retriever.vectorstore.add_documents(summary_docs)
    retriever.docstore.mset(
        [(doc.metadata[id_key], doc) for doc in parent_docs]
    )

    logger.info(
        "  Strategy 2 indexed. Summary proxy vectors: %d", vectorstore._collection.count()
    )
    return retriever



In [15]:


def build_mvr_strategy_3_hypothetical_questions(
    parent_docs: List[Document],
    questions: List[str],
    embeddings: HuggingFaceEmbeddings,
    config: MVRConfig,
) -> MultiVectorRetriever:
    """
    Strategy 3 -- Hypothetical Question Embeddings.

    Embed one LLM-generated hypothetical question per parent document.
    This aligns the embedding space with the way users actually phrase their
    queries, dramatically improving recall for question-style queries.

    The key insight: embedding spaces trained on Q&A pairs show that
    question-to-question cosine similarity is systematically higher than
    question-to-answer cosine similarity. By bridging the representation gap
    at ingestion time (pre-generating questions that the chunk answers), we
    ensure that user questions find their matching chunks reliably.

    Indexing steps:
        1. Wrap each generated question string as a Document with doc_id.
        2. Embed question Documents into a dedicated ChromaDB collection.
        3. Store full parent docs in InMemoryStore keyed by doc_id.

    Args:
        parent_docs : Parent documents with doc_id metadata.
        questions   : LLM-generated hypothetical question strings.
        embeddings  : HuggingFace embedding model.
        config      : MVRConfig.

    Returns:
        Populated MultiVectorRetriever for Strategy 3.
    """
    logger.info("=== Strategy 3: Hypothetical Questions ===")

    if len(questions) != len(parent_docs):
        raise ValueError(
            f"Mismatch: {len(parent_docs)} parent docs but {len(questions)} questions."
        )

    vectorstore = Chroma(
        collection_name=config.CHROMA_COLLECTIONS[MVRStrategy.HYPOTHETICAL_Q],
        embedding_function=embeddings,
        persist_directory=config.CHROMA_PERSIST_DIR,
    )
    docstore = InMemoryStore()
    id_key   = "doc_id"

    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key,
        search_kwargs={"k": config.RETRIEVER_K},
    )

    # Wrap each question as a Document carrying the parent's doc_id
    question_docs: List[Document] = [
        Document(
            page_content=question,
            metadata={
                id_key:       parent.metadata[id_key],
                "source":     parent.metadata.get("source", ""),
                "paper_name": parent.metadata.get("paper_name", ""),
                "proxy_type": "hypothetical_question",
            }
        )
        for parent, question in zip(parent_docs, questions)
    ]

    retriever.vectorstore.add_documents(question_docs)
    retriever.docstore.mset(
        [(doc.metadata[id_key], doc) for doc in parent_docs]
    )

    logger.info(
        "  Strategy 3 indexed. Question proxy vectors: %d", vectorstore._collection.count()
    )
    return retriever


In [16]:


# ===========================================================================
# SECTION 8 -- RAG CHAIN FACTORY
# A single factory function that wraps any MVR strategy retriever into
# a complete LCEL RAG chain. Strategy selection is fully decoupled from
# chain assembly -- any retriever plugs in identically.
# ===========================================================================

def build_rag_chain(
    retriever: MultiVectorRetriever,
    llm: AzureChatOpenAI,
    config: MVRConfig,
    strategy_name: str = "",
):
    """
    Build a complete LCEL RAG chain wrapping the given MultiVectorRetriever.

    Because MultiVectorRetriever is a standard LangChain Runnable, this
    factory function is completely agnostic to which strategy was used to
    build the retriever. The same chain assembly code works for all three
    strategies, which is a core benefit of the Runnable interface.

    Context formatting annotates each retrieved parent document with its
    source paper name, page number, and doc_id, giving the LLM explicit
    attribution information it can use in answers.

    Args:
        retriever     : Any populated MultiVectorRetriever.
        llm           : AzureChatOpenAI instance.
        config        : MVRConfig with RAG_PROMPT_TEMPLATE.
        strategy_name : Human-readable strategy label for logging.

    Returns:
        Tuple of (compiled LCEL chain, retriever).
    """
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT_TEMPLATE)

    def format_docs(docs: List[Document]) -> str:
        """
        Format retrieved parent documents into an annotated context block.
        Each doc is labeled with paper name, page, and doc_id so the LLM
        can attribute statements to specific sections of specific papers.
        """
        parts = []
        for i, doc in enumerate(docs, start=1):
            paper  = doc.metadata.get("paper_name", doc.metadata.get("source", "Unknown"))
            page   = doc.metadata.get("page", "?")
            doc_id = doc.metadata.get("doc_id", "")[:8]   # first 8 chars of UUID
            text   = doc.page_content.strip()
            parts.append(
                f"[Doc {i} | {paper} | Page {page} | id:{doc_id}]\n{text}"
            )
        return "\n\n" + ("-" * 60 + "\n").join(parts)

    rag_chain = (
        {
            "context":  retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    logger.info("RAG chain assembled for strategy: %s", strategy_name or "unknown")
    return rag_chain, retriever


In [17]:

# ===========================================================================
# SECTION 9 -- QUERY INTERFACE
# ===========================================================================

def query_mvr(
    chain,
    retriever: MultiVectorRetriever,
    question: str,
    strategy_label: str,
    show_sources: bool = True,
) -> str:
    """
    Execute a question against a Multi-Vector Retriever RAG chain.

    Prints retrieved parent document metadata for transparency, then returns
    the LLM-generated answer. The strategy_label in the header makes it easy
    to compare output quality across the three strategies for the same query.

    Args:
        chain          : Compiled LCEL RAG chain.
        retriever      : MultiVectorRetriever for source display.
        question       : Natural language question.
        strategy_label : Display label (e.g. "Strategy 2: Summaries").
        show_sources   : If True, print source metadata and excerpts.

    Returns:
        Answer string from Azure OpenAI.
    """
    logger.info("[%s] Query: '%s'", strategy_label, question)

    if show_sources:
        parent_docs = retriever.invoke(question)
        print("\n" + "=" * 72)
        print(f"STRATEGY : {strategy_label}")
        print(f"QUERY    : {question}")
        print("=" * 72)
        print(f"\nRETRIEVED PARENT DOCUMENTS ({len(parent_docs)}):")
        for i, doc in enumerate(parent_docs, start=1):
            paper   = doc.metadata.get("paper_name", "Unknown")
            page    = doc.metadata.get("page", "?")
            chars   = len(doc.page_content)
            snippet = doc.page_content[:250].replace("\n", " ").strip()
            print(f"\n  [{i}] {paper} | Page {page} | {chars} chars")
            print(f"       {snippet}...")

    answer = chain.invoke(question)

    if show_sources:
        print(f"\nANSWER:\n{answer}")
        print("=" * 72 + "\n")

    return answer


In [18]:

# ===========================================================================
# SECTION 10 -- COMPARATIVE RUNNER
# Run the same query through all three strategies and display results
# side by side for direct comparison.
# ===========================================================================

def compare_strategies(
    chains_and_retrievers: Dict[str, tuple],
    question: str,
) -> Dict[str, str]:
    """
    Run the same question through all three MVR strategies and collect answers.

    This comparative runner is invaluable during evaluation: it lets you
    observe how the same query produces different retrieved parent documents
    depending on whether the ChromaDB index holds raw sub-chunks, summaries,
    or hypothetical questions as proxy vectors.

    Args:
        chains_and_retrievers : Dict mapping strategy label to (chain, retriever).
        question              : The query to run across all strategies.

    Returns:
        Dict mapping strategy label to generated answer string.
    """
    results: Dict[str, str] = {}
    for label, (chain, retriever) in chains_and_retrievers.items():
        answer = query_mvr(chain, retriever, question, label, show_sources=True)
        results[label] = answer
    return results


In [19]:
"""
    End-to-end Multi-Vector Retriever RAG pipeline orchestrator.

    Execution order:
        1.  Download three arXiv PDFs (cached after first run).
        2.  Load and split into parent chunks with UUID doc_ids.
        3.  Initialize HuggingFace embedding model (local).
        4.  Initialize Azure OpenAI LLM (remote).
        5.  Generate LLM summaries for Strategy 2 (batch, concurrent).
        6.  Generate LLM hypothetical questions for Strategy 3 (batch, concurrent).
        7.  Build MultiVectorRetriever for each of the three strategies.
        8.  Assemble LCEL RAG chains.
        9.  Run comparative demo queries across all three strategies.

    Note on ingestion cost:
        Steps 5 and 6 each make one Azure OpenAI API call per parent chunk.
        With ~300 parent chunks from three arXiv papers and batch_concurrency=3,
        expect approximately 200 total API calls split across the two steps.
        Each call uses ~300 input tokens (parent chunk) + 200/100 output tokens.
        Estimated total cost at GPT-4o-mini pricing: < $0.10 USD.
        On second run, PDF download and LLM proxy generation are both skipped
        by loading from cached PDFs (ChromaDB is already persisted).
        InMemoryStore is always re-populated from cached PDFs (see production
        upgrade note in README for persistent docstore options).
    """

'\n    End-to-end Multi-Vector Retriever RAG pipeline orchestrator.\n\n    Execution order:\n        1.  Download three arXiv PDFs (cached after first run).\n        2.  Load and split into parent chunks with UUID doc_ids.\n        3.  Initialize HuggingFace embedding model (local).\n        4.  Initialize Azure OpenAI LLM (remote).\n        5.  Generate LLM summaries for Strategy 2 (batch, concurrent).\n        6.  Generate LLM hypothetical questions for Strategy 3 (batch, concurrent).\n        7.  Build MultiVectorRetriever for each of the three strategies.\n        8.  Assemble LCEL RAG chains.\n        9.  Run comparative demo queries across all three strategies.\n\n    Note on ingestion cost:\n        Steps 5 and 6 each make one Azure OpenAI API call per parent chunk.\n        With ~300 parent chunks from three arXiv papers and batch_concurrency=3,\n        expect approximately 200 total API calls split across the two steps.\n        Each call uses ~300 input tokens (parent chunk)

In [20]:
config = MVRConfig()


In [21]:
pdf_paths = download_pdfs(config)


2026-03-27 08:08:48 | INFO     | multi_vector_retriever | Cached PDF: attention_is_all_you_need.pdf (2163.3 KB)
2026-03-27 08:08:48 | INFO     | multi_vector_retriever | Cached PDF: bert_pretraining_transformers.pdf (757.0 KB)
2026-03-27 08:08:48 | INFO     | multi_vector_retriever | Cached PDF: rag_knowledge_intensive_nlp.pdf (864.6 KB)


In [22]:
parent_docs = load_and_chunk_documents(pdf_paths, config)


2026-03-27 08:08:48 | INFO     | multi_vector_retriever | Loading: attention_is_all_you_need.pdf
2026-03-27 08:08:50 | INFO     | multi_vector_retriever |   attention_is_all_you_need.pdf --> 40 parent chunks
2026-03-27 08:08:50 | INFO     | multi_vector_retriever | Loading: bert_pretraining_transformers.pdf
2026-03-27 08:08:52 | INFO     | multi_vector_retriever |   bert_pretraining_transformers.pdf --> 68 parent chunks
2026-03-27 08:08:52 | INFO     | multi_vector_retriever | Loading: rag_knowledge_intensive_nlp.pdf
2026-03-27 08:08:53 | INFO     | multi_vector_retriever |   rag_knowledge_intensive_nlp.pdf --> 70 parent chunks
2026-03-27 08:08:53 | INFO     | multi_vector_retriever | Total parent chunks across all PDFs: 178 (avg ~1039 chars each)


In [23]:
embeddings = build_embeddings(config)

2026-03-27 08:08:53 | INFO     | multi_vector_retriever | Embedding model: sentence-transformers/all-MiniLM-L6-v2 (device=cpu)
2026-03-27 08:08:53 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


C:\Users\dhanu\AppData\Local\Temp\ipykernel_15076\3860197859.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


2026-03-27 08:08:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-27 08:08:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-27 08:08:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-27 08:08:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-27 08:08:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_trans

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2520.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-03-27 08:08:56 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-27 08:08:56 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-27 08:08:57 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-27 08:08:57 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-27 08:08:57 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=fals

In [24]:
llm = build_azure_llm(config)

2026-03-27 08:08:58 | INFO     | multi_vector_retriever | Azure OpenAI: deployment='gpt-4.1'  api_version='2024-12-01-preview'


In [28]:
# Reduce concurrency to avoid rate limits (processing 42 documents)
import time
original_concurrency = config.BATCH_CONCURRENCY
config.BATCH_CONCURRENCY = 1  # Process one at a time to avoid rate limiting

print(f"Generating summaries for {len(parent_docs)} documents (concurrency=1)...")
summaries  = generate_summaries(parent_docs, llm, config)
print(f"✓ Generated {len(summaries)} summaries")


Generating summaries for 178 documents (concurrency=1)...
2026-03-27 08:46:40 | INFO     | multi_vector_retriever | Generating summaries for 178 parent docs ...
2026-03-27 08:46:43 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:46:45 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:46:47 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:46:50 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:46:52 | INFO 

In [30]:
# Wait and generate hypothetical questions with same reduced concurrency
time.sleep(10)  # Add delay between batch operations
print(f"Generating questions for {len(parent_docs)} documents (concurrency=1)...")
questions  = generate_hypothetical_questions(parent_docs, llm, config)
config.BATCH_CONCURRENCY = original_concurrency  # Restore original setting
print(f"✓ Generated {len(questions)} questions")


Generating questions for 178 documents (concurrency=1)...
2026-03-27 08:54:33 | INFO     | multi_vector_retriever | Generating hypothetical questions for 178 parent docs ...
2026-03-27 08:54:37 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:54:39 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:54:40 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:54:42 | INFO     | httpx | HTTP Request: POST https://dhanush-ai507.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-27 08:

In [31]:
# Step 7: Build all three MultiVectorRetriever instances
retriever_s1 = build_mvr_strategy_1_small_chunks(parent_docs, embeddings, config)
retriever_s2 = build_mvr_strategy_2_summaries(parent_docs, summaries, embeddings, config)
retriever_s3 = build_mvr_strategy_3_hypothetical_questions(parent_docs, questions, embeddings, config)


2026-03-27 08:59:20 | INFO     | multi_vector_retriever | === Strategy 1: Small Chunks ===
2026-03-27 08:59:21 | INFO     | multi_vector_retriever |   Parent docs: 178  |  Child chunks: 1131  (avg 6.4 children/parent)
2026-03-27 08:59:34 | INFO     | multi_vector_retriever |   Strategy 1 indexed. ChromaDB vectors: 2262
2026-03-27 08:59:34 | INFO     | multi_vector_retriever | === Strategy 2: Summary Embeddings ===
2026-03-27 08:59:38 | INFO     | multi_vector_retriever |   Strategy 2 indexed. Summary proxy vectors: 178
2026-03-27 08:59:38 | INFO     | multi_vector_retriever | === Strategy 3: Hypothetical Questions ===
2026-03-27 08:59:40 | INFO     | multi_vector_retriever |   Strategy 3 indexed. Question proxy vectors: 178


In [33]:
# Step 8: Build LCEL RAG chains (one per strategy)
chain_s1, _ = build_rag_chain(retriever_s1, llm, config, "Strategy 1: Small Chunks")
chain_s2, _ = build_rag_chain(retriever_s2, llm, config, "Strategy 2: Summaries")
chain_s3, _ = build_rag_chain(retriever_s3, llm, config, "Strategy 3: Hypothetical Questions")


2026-03-27 08:59:49 | INFO     | multi_vector_retriever | RAG chain assembled for strategy: Strategy 1: Small Chunks
2026-03-27 08:59:49 | INFO     | multi_vector_retriever | RAG chain assembled for strategy: Strategy 2: Summaries
2026-03-27 08:59:49 | INFO     | multi_vector_retriever | RAG chain assembled for strategy: Strategy 3: Hypothetical Questions


In [34]:
chains_and_retrievers = {
    "Strategy 1: Small Chunks":          (chain_s1, retriever_s1),
    "Strategy 2: Summaries":             (chain_s2, retriever_s2),
    "Strategy 3: Hypothetical Questions":(chain_s3, retriever_s3),
}

In [35]:
# Step 9: Demo queries -- run each through all three strategies
demo_questions = [
    # Tests conceptual understanding (favors Strategy 2: Summaries)
    "What is the core intuition behind the Transformer architecture and why does it replace recurrence?",

    # Tests precise technical retrieval (favors Strategy 1: Small Chunks)
    "What are the dimensions of the key, query, and value matrices in the Transformer's attention mechanism?",

    # Tests question-style retrieval (favors Strategy 3: Hypothetical Questions)
    "How does BERT's pre-training differ from GPT-style left-to-right language modeling?",

    # Cross-paper synthesis question
    "How does the RAG model use a retriever and generator together during inference?",
]

In [36]:
for question in demo_questions:
    compare_strategies(chains_and_retrievers, question)


2026-03-27 08:59:53 | INFO     | multi_vector_retriever | [Strategy 1: Small Chunks] Query: 'What is the core intuition behind the Transformer architecture and why does it replace recurrence?'

STRATEGY : Strategy 1: Small Chunks
QUERY    : What is the core intuition behind the Transformer architecture and why does it replace recurrence?

RETRIEVED PARENT DOCUMENTS (2):

  [1] Attention Is All You Need | Page 1 | 1104 chars
       constraint of sequential computation, however, remains. Attention mechanisms have become an integral part of compelling sequence modeling and transduc- tion models in various tasks, allowing modeling of dependencies without regard to their distance i...

  [2] Attention Is All You Need | Page 0 | 1146 chars
       Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com 